In [1]:
from langchain_openai import AzureChatOpenAI
from langchain.agents import AgentType
import re

In [2]:
# Load environment variables from .env file
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Get environment variables for Azure OpenAI
AZURE_OPENAI_ENDPOINT_EMBEDDING = os.getenv('AZURE_OPENAI_ENDPOINT_EMBEDDING')
AZURE_OPENAI_API_KEY_EMBEDDING = os.getenv('AZURE_OPENAI_API_KEY_EMBEDDING')

AZURE_OPENAI_ENDPOINT_LLM = os.getenv('AZURE_OPENAI_ENDPOINT_LLM')
AZURE_OPENAI_API_KEY_LLM = os.getenv('AZURE_OPENAI_API_KEY_LLM')

LLM_MODEL_ID = os.getenv('LLM_MODEL_ID')
EMBEDDING_MODEL_ID = os.getenv('EMBEDDING_MODEL_ID')

# Print loaded values (masking sensitive information)
print("Environment variables loaded:")
print(f"AZURE_OPENAI_ENDPOINT_LLM: {AZURE_OPENAI_ENDPOINT_LLM}")
print(f"AZURE_OPENAI_ENDPOINT_EMBEDDING: {AZURE_OPENAI_ENDPOINT_EMBEDDING}")
print(f"LLM_MODEL_ID: {LLM_MODEL_ID}")
print(f"EMBEDDING_MODEL_ID: {EMBEDDING_MODEL_ID}")
print(f"API Keys loaded: {'✓' if AZURE_OPENAI_API_KEY_LLM and AZURE_OPENAI_API_KEY_EMBEDDING else '✗'}")

Environment variables loaded:
AZURE_OPENAI_ENDPOINT_LLM: https://ractopenaitest.openai.azure.com/
AZURE_OPENAI_ENDPOINT_EMBEDDING: https://ractopenaitest.openai.azure.com/
LLM_MODEL_ID: gpt-4
EMBEDDING_MODEL_ID: text-embedding-ada-002
API Keys loaded: ✓


In [3]:
def llm():
    parameters = {
        "max_tokens": 256,  # Maximum number of tokens in the generated output
        "temperature": 0.5, # Controls randomness/creativity of responses
    }
    
    llm_model = AzureChatOpenAI(
        azure_endpoint=AZURE_OPENAI_ENDPOINT_LLM,
        api_key=AZURE_OPENAI_API_KEY_LLM,
        model=LLM_MODEL_ID,
        api_version="2023-05-15",  # Use the correct API version for your Azure OpenAI deployment
        **parameters
    )
    return llm_model

In [7]:
def add_numbers(inputs:str) -> dict:
    """
    Adds a list of numbers provided in the input dictionary or extracts numbers from a string.

    Parameters:
    - inputs (str): 
    string, it should contain numbers that can be extracted and summed.

    Returns:
    - dict: A dictionary with a single key "result" containing the sum of the numbers.

    Example Input (Dictionary):
    {"numbers": [10, 20, 30]}

    Example Input (String):
    "Add the numbers 10, 20, and 30."

    Example Output:
    {"result": 60}
    """
    numbers = [int(x) for x in inputs.replace(",", "").split() if x.isdigit()]

    
    result = sum(numbers)
    return {"result": result}

In [8]:
add_numbers("1 2") 

{'result': 3}

In [4]:
model = llm()

In [6]:
response = model.invoke("What is tool calling in langchain?")
print("\nResponse Content: ", response.content)


Response Content:  In Langchain, "tool calling" refers to the process of integrating external tools or APIs into a language model to enhance its capabilities. This allows the language model to perform tasks that require specific functionalities or access to external data sources, such as making API calls, querying databases, or using specialized algorithms. Tool calling enables the language model to provide more accurate and contextually relevant responses by leveraging these external resources.


In [10]:
# tool
from langchain.agents import Tool
add_tool = Tool (
    name="AddTool",
    func=add_numbers,
    description="Add a list of number and returns the result."
)

print("tool object", add_tool)

tool object name='AddTool' description='Add a list of number and returns the result.' func=<function add_numbers at 0x00000295789467A0>


In [11]:
print("Tool Name:")
print(add_tool.name)

# Tool description
print("Tool Description:")
print(add_tool.description)

# Tool function
print("Tool Function:")
print(add_tool.invoke)

Tool Name:
AddTool
Tool Description:
Add a list of number and returns the result.
Tool Function:
<bound method BaseTool.invoke of Tool(name='AddTool', description='Add a list of number and returns the result.', func=<function add_numbers at 0x00000295789467A0>)>


In [12]:
print("Calling Tool Function:")
test_input = "10 20 30 a b" 
print(add_tool.invoke(test_input))  # Example

Calling Tool Function:
{'result': 60}


@tool operator
Now you know how to create a tool with a Tool class (using Tool Interface), there's actually another way that you can create a tool using @tool decorator. The recommended way to create tools is using the @tool decorator. This decorator is designed to simplify the process of tool creation and should be used in most cases. After defining a function, you can decorate it with @tool to create a tool that implements the Tool Interface.

@tool opertor makes tools out of functions. See below:

In [13]:
from langchain_core.tools import tool
import re

@tool
def add_numbers(inputs:str) -> dict:
    """
    Adds a list of numbers provided in the input string.
    Parameters:
    - inputs (str): 
    string, it should contain numbers that can be extracted and summed.
    Returns:
    - dict: A dictionary with a single key "result" containing the sum of the numbers.
    Example Input:
    "Add the numbers 10, 20, and 30."
    Example Output:
    {"result": 60}
    """
    # Use regular expressions to extract all numbers from the input
    numbers = [int(num) for num in re.findall(r'\d+', inputs)]
    # numbers = [int(x) for x in inputs.replace(",", "").split() if x.isdigit()]
    
    result = sum(numbers)
    return {"result": result}

In [14]:
print("Name: \n", add_numbers.name)
print("Description: \n", add_numbers.description) 
print("Args: \n", add_numbers.args) 

Name: 
 add_numbers
Description: 
 Adds a list of numbers provided in the input string.
Parameters:
- inputs (str): 
string, it should contain numbers that can be extracted and summed.
Returns:
- dict: A dictionary with a single key "result" containing the sum of the numbers.
Example Input:
"Add the numbers 10, 20, and 30."
Example Output:
{"result": 60}
Args: 
 {'inputs': {'title': 'Inputs', 'type': 'string'}}


In [15]:
test_input = "what is the sum between 10, 20 and 30 " 
print(add_numbers.invoke(test_input))  

{'result': 60}


In [16]:
# Comparing the two approaches
print("Tool Constructor Approach:")

print(f"Has Schema: {hasattr(add_tool, 'args_schema')}")
print("\n")

print("@tool Decorator Approach:")


print(f"Has Schema: {hasattr(add_numbers, 'args_schema')}")
print(f"Args Schema Info: {add_numbers.args}")

Tool Constructor Approach:
Has Schema: True


@tool Decorator Approach:
Has Schema: True
Args Schema Info: {'inputs': {'title': 'Inputs', 'type': 'string'}}


In [18]:
from typing import List

@tool
def add_numbers_with_options(numbers: List[float], absolute: bool = False) -> float:
    """
    Adds a list of numbers provided as input.

    Parameters:
    - numbers (List[float]): A list of numbers to be summed.
    - absolute (bool): If True, use the absolute values of the numbers before summing.

    Returns:
    - float: The total sum of the numbers.
    """
    if absolute:
        numbers = [abs(n) for n in numbers]
    return sum(numbers)

In [19]:
print(f"Args Schema Info: {add_numbers_with_options.args}")
print(f"Args Schema Info: {add_numbers.args}")

Args Schema Info: {'numbers': {'title': 'Numbers', 'type': 'array', 'items': {'type': 'number'}}, 'absolute': {'title': 'Absolute', 'default': False, 'type': 'boolean'}}
Args Schema Info: {'inputs': {'title': 'Inputs', 'type': 'string'}}


In [20]:
print(add_numbers_with_options.invoke({"numbers":[-1.1,-2.1,-3.0],"absolute":False}))
print(add_numbers_with_options.invoke({"numbers":[-1.1,-2.1,-3.0],"absolute":True}))

-6.2
6.2


In [23]:
from typing import Dict, Union 
@tool
def sum_numbers_with_complex_output(inputs: str) -> Dict[str, Union[float, str]]:
    """
    Extracts and sums all integers and decimal numbers from the input string.

    Parameters:
    - inputs (str): A string that may contain numeric values.

    Returns:
    - dict: A dictionary with the key "result". If numbers are found, the value is their sum (float). 
            If no numbers are found or an error occurs, the value is a corresponding message (str).

    Example Input:
    "Add 10, 20.5, and -3."

    Example Output:
    {"result": 27.5}
    """
    matches = re.findall(r'-?\d+(?:\.\d+)?', inputs)
    if not matches:
        return {"result": "No numbers found in input."}
    try:
        numbers = [float(num) for num in matches]
        total = sum(numbers)
        return {"result": total}
    except Exception as e:
        return {"result": f"Error during summation: {str(e)}"}

In [24]:
@tool
def sum_numbers_from_text(inputs: str) -> float:
    """
    Adds a list of numbers provided in the input string.
    
    Args:
        text: A string containing numbers that should be extracted and summed.
        
    Returns:
        The sum of all numbers found in the input.
    """
    # Use regular expressions to extract all numbers from the input
    numbers = [int(num) for num in re.findall(r'\d+', inputs)]
    result = sum(numbers)
    return result

nitialize_agent
When you set up an agent, you're connecting tools and an LLM to work together seamlessly. The agent uses the LLM to understand what needs to be done and decides which tool to use based on the task. Here's a simple overview of the key parts:

Relationship between agent and LLM
The agent acts as the decision-maker, figuring out which tools to use and when.
The LLM is the reasoning engine. It:
Interprets the user's input.
Helps the agent make decisions.
Generates a response based on the output of the tools.
Think of the agent as the manager assigning tasks and the LLM as the brain solving problems or delegating work.

Key parameters of initialize_agent
tools- see above

llm see above

agent:

Specifies the reasoning framework for the agent.
"zero-shot-react-description" enables:
Zero-shot reasoning: The agent can solve tasks it hasn't seen before by thinking through the problem step by step.
React framework: A logical loop of:
Reason → Think about the task.
Act → Use a tool to perform an action.
Observe → Check the tool's output.
Plan → Decide what to do next.
verbose:

If True, it prints detailed logs of the agent’s thought process.
Useful for debugging or understanding how the agent makes decisions.

In [25]:
from langchain.agents import initialize_agent
agent = initialize_agent ([add_tool], model, agent="zero-shot-react-description", verbose=True, handle_parsing_errors=True)

C:\Users\madusankas\AppData\Local\Temp\ipykernel_26156\3280630464.py:2: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 0.3.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  agent = initialize_agent ([add_tool], model, agent="zero-shot-react-description", verbose=True, handle_parsing_errors=True)


In [26]:
# Use the agent
response =agent.run("In 2023, the US GDP was approximately $27.72 trillion, while Canada's was around $2.14 trillion and Mexico's was about $1.79 trillion what is the total.")

C:\Users\madusankas\AppData\Local\Temp\ipykernel_26156\1910783247.py:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 0.3.0. Use invoke instead.
  response =agent.run("In 2023, the US GDP was approximately $27.72 trillion, while Canada's was around $2.14 trillion and Mexico's was about $1.79 trillion what is the total.")




> Entering new AgentExecutor chain...
I need to add the GDPs of the US, Canada, and Mexico to find the total GDP.

Action: AddTool
Action Input: "27.72, 2.14, 1.79"
Observation: {'result': 0}
Thought:I need to add the GDPs of the US, Canada, and Mexico to find the total GDP.

Action: AddTool
Action Input: "27.72, 2.14, 1.79"
Observation: {'result': 0}
Thought:The result from the AddTool seems incorrect as it returned 0, which is not possible given the inputs. I need to add the numbers manually.

Action: AddTool
Action Input: "27.72, 2.14, 1.79"
Observation: {'result': 0}
Thought:The result from the AddTool seems incorrect as it returned 0, which is not possible given the inputs. I need to add the numbers manually.

Action: AddTool
Action Input: "27.72, 2.14, 1.79"
Observation: {'result': 0}
Thought:The AddTool is not functioning correctly as it keeps returning 0 despite different inputs. I will manually calculate the total GDP.

Final Answer: The total GDP of the US, Canada, and Mexi

In [27]:
response

'The total GDP of the US, Canada, and Mexico in 2023 is approximately $31.65 trillion.'

In [28]:
agent.invoke({"input": "Add 10, 20, two and 30"})



> Entering new AgentExecutor chain...
The numbers given are 10, 20, and 30. "Two" is written in word form and should be converted to the number 2.

Action: AddTool
Action Input: "10, 20, 2, 30"
Observation: {'result': 62}
Thought:The numbers given are 10, 20, and 30. "Two" is written in word form and should be converted to the number 2.

Action: AddTool
Action Input: "10, 20, 2, 30"
Observation: {'result': 62}
Thought:I now know the final answer
Final Answer: 62

> Finished chain.
I now know the final answer
Final Answer: 62

> Finished chain.


{'input': 'Add 10, 20, two and 30', 'output': '62'}

In [29]:
agent_2 = initialize_agent([sum_numbers_from_text], model, agent="structured-chat-zero-shot-react-description", verbose=True, handle_parsing_errors=True)
response = agent_2.invoke({"input": "Add 10, 20 and 30"})
print(response)



> Entering new AgentExecutor chain...
{
  "action": "sum_numbers_from_text",
  "action_input": {
    "text": "10, 20 and 30"
  }
}

> Finished chain.
{'input': 'Add 10, 20 and 30', 'output': '{\n  "action": "sum_numbers_from_text",\n  "action_input": {\n    "text": "10, 20 and 30"\n  }\n}'}
{
  "action": "sum_numbers_from_text",
  "action_input": {
    "text": "10, 20 and 30"
  }
}

> Finished chain.
{'input': 'Add 10, 20 and 30', 'output': '{\n  "action": "sum_numbers_from_text",\n  "action_input": {\n    "text": "10, 20 and 30"\n  }\n}'}


In [30]:
agent_3 = initialize_agent([sum_numbers_with_complex_output], model, agent="openai-functions", verbose=True, handle_parsing_errors=True)
response = agent_3.invoke({"input": "Add 10, 20 and 30"})
print(response)



> Entering new AgentExecutor chain...

Invoking: `sum_numbers_with_complex_output` with `{'inputs': 'Add 10, 20 and 30'}`


{'result': 60.0}
Invoking: `sum_numbers_with_complex_output` with `{'inputs': 'Add 10, 20 and 30'}`


{'result': 60.0}The sum of 10, 20, and 30 is 60.

> Finished chain.
{'input': 'Add 10, 20 and 30', 'output': 'The sum of 10, 20, and 30 is 60.'}
The sum of 10, 20, and 30 is 60.

> Finished chain.
{'input': 'Add 10, 20 and 30', 'output': 'The sum of 10, 20, and 30 is 60.'}


In [31]:
agent_2 = initialize_agent(
    [add_numbers_with_options],
    model,
    agent="structured-chat-zero-shot-react-description",
    verbose=True
)

In [32]:
response = agent_2.invoke({
    "input": "Add -10, -20, and -30 using absolute values."
})
print(response)



> Entering new AgentExecutor chain...
Action:
```json
{
  "action": "add_numbers_with_options",
  "action_input": {
    "numbers": [-10, -20, -30],
    "absolute": true
  }
}
```
Observation: 60.0
Thought:Action:
```json
{
  "action": "add_numbers_with_options",
  "action_input": {
    "numbers": [-10, -20, -30],
    "absolute": true
  }
}
```
Observation: 60.0
Thought:Action:
```json
{
  "action": "Final Answer",
  "action_input": "The sum of -10, -20, and -30 using absolute values is 60.0."
}
```

> Finished chain.
{'input': 'Add -10, -20, and -30 using absolute values.', 'output': 'The sum of -10, -20, and -30 using absolute values is 60.0.'}
Action:
```json
{
  "action": "Final Answer",
  "action_input": "The sum of -10, -20, and -30 using absolute values is 60.0."
}
```

> Finished chain.
{'input': 'Add -10, -20, and -30 using absolute values.', 'output': 'The sum of -10, -20, and -30 using absolute values is 60.0.'}
